# SVM Regression

California housing with linear, poly, RBF, and sigmoid kernels.

## Preprocessing

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn as sk
from sklearn.model_selection import train_test_split
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler

data_df = pd.read_csv("housing.csv")
data_df['total_bedrooms'] = data_df['total_bedrooms'].fillna(data_df['total_bedrooms'].median())
data_encoded = pd.get_dummies(data_df, columns=['ocean_proximity'], dtype=float)

data = data_encoded.to_numpy()
target_col = data_encoded.columns.get_loc('median_house_value')
feature_cols = [i for i in range(data.shape[1]) if i != target_col]
x_data = data[:, feature_cols]
y = data[:, target_col]
feature_names = [data_encoded.columns[i] for i in feature_cols]

scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_data)

X_train, X_test, y_train, y_test = train_test_split(
    x_scaled, y, test_size=0.33, shuffle=True, random_state=42
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## Train SVR with each kernel

Subsample the training set — full SVR on 13k rows is too slow.

In [ ]:
np.random.seed(42)
sample_size = 5000
sample_idx = np.random.choice(X_train.shape[0], sample_size, replace=False)
X_train_sample = X_train[sample_idx]
y_train_sample = y_train[sample_idx]

kernels = ['linear', 'poly', 'rbf', 'sigmoid']
models = {}
predictions = {}

for kernel in kernels:
    print(f"Training {kernel}...")
    svr = SVR(kernel=kernel, C=100000, epsilon=10000)
    svr.fit(X_train_sample, y_train_sample)
    models[kernel] = svr
    predictions[kernel] = svr.predict(X_test)
    print("  done.")

## Predicted vs actual per kernel

In [ ]:
fig, axs = plt.subplots(4, 1, figsize=(18, 28))

for i, kernel in enumerate(kernels):
    y_pred = predictions[kernel]

    mse = np.round((1 / X_test.shape[0]) * np.sum((y_pred - y_test) ** 2), 2)
    mae = np.round((1 / X_test.shape[0]) * np.sum(np.abs(y_pred - y_test)), 2)
    r2 = np.round(sk.metrics.r2_score(y_test, y_pred), 4)

    axs[i].scatter(range(X_test.shape[0]), y_pred, s=5, alpha=0.3, label='Predicted')
    axs[i].scatter(range(X_test.shape[0]), y_test, s=5, alpha=0.3, label='Actual')
    axs[i].set_title(kernel)
    axs[i].set_xlabel('Sample')
    axs[i].set_ylabel('House Value ($)')
    axs[i].legend()

    errors = f"MSE: {mse}\nMAE: {mae}\nR2: {r2}"
    axs[i].text(0.01, 0.95, errors, transform=axs[i].transAxes, verticalalignment='top',
               fontsize=11, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

## Best kernel by feature

In [ ]:
best_kernel = max(predictions.keys(), key=lambda k: sk.metrics.r2_score(y_test, predictions[k]))
best_pred = predictions[best_kernel]
print(f"Best kernel: {best_kernel}")

plot_features = ['median_income', 'housing_median_age', 'latitude', 'longitude']
plot_indices = [feature_names.index(f) for f in plot_features]

fig, axs = plt.subplots(2, 2, figsize=(20, 16))

for idx, (feat, col_idx) in enumerate(zip(plot_features, plot_indices)):
    row, col = idx // 2, idx % 2
    axs[row, col].scatter(X_test[:, col_idx], y_test, alpha=0.15, s=5, label='Actual')
    axs[row, col].scatter(X_test[:, col_idx], best_pred, alpha=0.15, s=5, label='Predicted', color='coral')
    axs[row, col].set_xlabel(feat)
    axs[row, col].set_ylabel('median_house_value')
    axs[row, col].legend()

plt.tight_layout()
plt.show()

## Error distributions

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(18, 14))

for i, kernel in enumerate(kernels):
    row, col = i // 2, i % 2
    errors = predictions[kernel] - y_test
    axs[row, col].hist(errors, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
    axs[row, col].set_title(kernel)
    axs[row, col].set_xlabel('Error ($)')
    axs[row, col].set_ylabel('Frequency')
    axs[row, col].axvline(x=0, color='red', linestyle='--', linewidth=2)

plt.tight_layout()
plt.show()

## 3D geographic view

In [ ]:
long_idx = feature_names.index('longitude')
lat_idx = feature_names.index('latitude')

fig = plt.figure(figsize=(16, 10))
ax = fig.add_subplot(projection='3d')

ax.scatter3D(X_test[:, long_idx], X_test[:, lat_idx], best_pred,
             c=best_pred, cmap='Reds', alpha=0.3, s=5, label='Predicted')
ax.scatter3D(X_test[:, long_idx], X_test[:, lat_idx], y_test,
             c=y_test, cmap='Greens', alpha=0.3, s=5, label='Actual')

ax.set_xlabel('Longitude (z)')
ax.set_ylabel('Latitude (z)')
ax.set_zlabel('Median House Value ($)')
ax.legend()
ax.view_init(20, 30)
plt.tight_layout()
plt.show()